In [6]:
import json
from pathlib import Path

LOG_PATH = Path("logfile.txt")


def parse_json_stream(text: str):
    """Parses sequential JSON objects from a text stream."""
    decoder = json.JSONDecoder()
    idx = 0
    text_len = len(text)
    while idx < text_len:
        while idx < text_len and text[idx].isspace():
            idx += 1
        if idx >= text_len:
            break
        obj, end = decoder.raw_decode(text, idx)
        yield obj
        idx = end


def matched_people_in_record(record: dict) -> int:
    """Counts unique matched persons inside one comparison record."""
    matches = record.get("matches", []) or []
    if not matches:
        return int(record.get("matchesCount", 0)) * 2

    people = set()
    for match in matches:
        tree1_person_id = ((match.get("tree1") or {}).get("personId"))
        tree2_person_id = ((match.get("tree2") or {}).get("personId"))
        if tree1_person_id is not None:
            people.add(("tree1", str(tree1_person_id)))
        if tree2_person_id is not None:
            people.add(("tree2", str(tree2_person_id)))
    return len(people)


raw_log = LOG_PATH.read_text(encoding="utf-8")
records = list(parse_json_stream(raw_log))

# Tree-level metrics (one log record = one compared tree pair).
trees_total = len(records)
matched_trees = sum(1 for rec in records if int(rec.get("matchesCount", 0)) > 0)
matched_trees_share = (matched_trees / trees_total) if trees_total else 0.0

# Person-level metrics aggregated across all comparisons from logs.
persons_total = sum(int(rec.get("tree1Size", 0)) + int(rec.get("tree2Size", 0)) for rec in records)
matched_persons = sum(matched_people_in_record(rec) for rec in records)
matched_persons_share = (matched_persons / persons_total) if persons_total else 0.0

metrics = {
    "число деревьев": trees_total,
    "число совпавших деревьев": matched_trees,
    "доля совпавших деревьев": round(matched_trees_share, 6),
    "число персон тотал": persons_total,
    "число совпавших персон": matched_persons,
    "доля совпавших персон": round(matched_persons_share, 6),
}

metrics


{'число деревьев': 2000,
 'число совпавших деревьев': 2,
 'доля совпавших деревьев': 0.001,
 'число персон тотал': 2211307,
 'число совпавших персон': 96,
 'доля совпавших персон': 4.3e-05}

In [7]:
import json
from IPython.display import HTML, display


def pick_person_view(person: dict) -> dict:
    """Compact person payload for readable side-by-side output."""
    return {
        "_id": (person.get("_id") or {}).get("$oid", person.get("_id")),
        "surname": (person.get("surname") or [None])[0],
        "name": (person.get("name") or [None])[0],
        "middleName": (person.get("middleName") or [None])[0],
        "maidenName": (person.get("maidenName") or [None])[0],
        "birthdate": person.get("birthdate"),
        "birthplace": (person.get("birthplace") or [None])[0],
    }


matched_records = [rec for rec in records if int(rec.get("matchesCount", 0)) > 0]

print(f"Совпавших пар деревьев: {len(matched_records)}")

for idx, rec in enumerate(matched_records, start=1):
    first_match = rec["matches"][0]

    left = {
        "treeId": first_match["tree1"]["id"],
        "personId": first_match["tree1"]["personId"],
        "score": first_match["score"],
        "person": pick_person_view(first_match["tree1"]["person"]),
    }
    right = {
        "treeId": first_match["tree2"]["id"],
        "personId": first_match["tree2"]["personId"],
        "score": first_match["score"],
        "person": pick_person_view(first_match["tree2"]["person"]),
    }

    html = f"""
    <h4>Пара деревьев {idx}</h4>
    <table style='width:100%; border-collapse:collapse;' border='1'>
      <thead>
        <tr>
          <th style='padding:8px; text-align:left; width:50%;'>tree1 (левая)</th>
          <th style='padding:8px; text-align:left; width:50%;'>tree2 (правая)</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td style='vertical-align:top; padding:8px;'><pre>{json.dumps(left, ensure_ascii=False, indent=2)}</pre></td>
          <td style='vertical-align:top; padding:8px;'><pre>{json.dumps(right, ensure_ascii=False, indent=2)}</pre></td>
        </tr>
      </tbody>
    </table>
    """
    display(HTML(html))

Совпавших пар деревьев: 2


tree1 (левая),tree2 (правая)
"{ ""treeId"": ""5f6a051eed122bb214536e02"", ""personId"": ""5f6a051eed122bb214536e03"", ""score"": 100.0, ""person"": { ""_id"": ""5f6a051eed122bb214536e03"", ""surname"": ""sha256:e39cbccc62f39ff4ec255e13c2ae6686272144929fa6f7121fc5d28408d09ffc"", ""name"": ""sha256:4127efd16f0511e7e5cc857a4534a2714649570f4e9bbc5856e5436ae44588fd"", ""middleName"": ""sha256:00db11095e6d88891e0084144427a852f5ead9d9e06647dc173abeb3f3db7e8c"", ""maidenName"": null, ""birthdate"": ""sha256:3c13c4e0ccca338f153d2dfd85774f1f614d6e8d36752cfbe9624b82555713ee"", ""birthplace"": ""sha256:3df3480f1ee77c02baa72e5379e521c596dbe81e110bc63997d5923c515b831a"" } }","{ ""treeId"": ""6811f97abe100fecbec48421"", ""personId"": ""6811f97abe100fecbec48283"", ""score"": 100.0, ""person"": { ""_id"": ""6811f97abe100fecbec48283"", ""surname"": ""sha256:e39cbccc62f39ff4ec255e13c2ae6686272144929fa6f7121fc5d28408d09ffc"", ""name"": ""sha256:4127efd16f0511e7e5cc857a4534a2714649570f4e9bbc5856e5436ae44588fd"", ""middleName"": ""sha256:00db11095e6d88891e0084144427a852f5ead9d9e06647dc173abeb3f3db7e8c"", ""maidenName"": null, ""birthdate"": ""sha256:3c13c4e0ccca338f153d2dfd85774f1f614d6e8d36752cfbe9624b82555713ee"", ""birthplace"": ""sha256:3df3480f1ee77c02baa72e5379e521c596dbe81e110bc63997d5923c515b831a"" } }"


tree1 (левая),tree2 (правая)
"{ ""treeId"": ""5fec181a99821956bb1220ed"", ""personId"": ""601ae1146a77c589a5321d0b"", ""score"": 96.43, ""person"": { ""_id"": ""601ae1146a77c589a5321d0b"", ""surname"": ""sha256:043a9c7cd973b460f3c39168d9f068923b765655577eada078df010e5b8eacb0"", ""name"": ""sha256:cb2df79a767aaf2eea1e4336636da6231d8934cb73b20a5a4e1d8946ca98339d"", ""middleName"": ""sha256:7af86a9f3f68a072d7d07db20471ee1f5f6b98b33579bb9066975c286f3fbc28"", ""maidenName"": ""sha256:fbe56564cd874e454bb548b7e5e3dd4ea234d8d1b4c3bad58f255c14879ca21f"", ""birthdate"": ""sha256:fec4fcf3f1f1cd1b774664b33bd5274c5468eda2ecf328db6836659f5d854654"", ""birthplace"": ""sha256:7fabb78f5378be3a1e69d7b5463e9c20ff8c531b94af723b846b583f897b2b18"" } }","{ ""treeId"": ""605b71392bb5e94f65cba306"", ""personId"": ""605b713a2bb5e94f65cba31f"", ""score"": 96.43, ""person"": { ""_id"": ""605b713a2bb5e94f65cba31f"", ""surname"": ""sha256:043a9c7cd973b460f3c39168d9f068923b765655577eada078df010e5b8eacb0"", ""name"": ""sha256:cb2df79a767aaf2eea1e4336636da6231d8934cb73b20a5a4e1d8946ca98339d"", ""middleName"": ""sha256:7af86a9f3f68a072d7d07db20471ee1f5f6b98b33579bb9066975c286f3fbc28"", ""maidenName"": ""sha256:fbe56564cd874e454bb548b7e5e3dd4ea234d8d1b4c3bad58f255c14879ca21f"", ""birthdate"": ""sha256:fec4fcf3f1f1cd1b774664b33bd5274c5468eda2ecf328db6836659f5d854654"", ""birthplace"": ""sha256:7fabb78f5378be3a1e69d7b5463e9c20ff8c531b94af723b846b583f897b2b18"" } }"
